In [1]:
import os
# os.environ['http_proxy'] = "http://localhost:7890"
# os.environ['https_proxy'] = "http://localhost:7890"
# os.environ['all_proxy'] = "socks5://localhost:7890"
os.environ['HF_ENDPOINT'] = "https://hf-mirror.com"

## (1) Load model

In [2]:
from model import Mamba, ModelArgs
from transformers import AutoTokenizer

# One of:
#     'state-spaces/mamba-2.8b-slimpj'
#     'state-spaces/mamba-2.8b'
#     'state-spaces/mamba-1.4b'
#     'state-spaces/mamba-790m'
#     'state-spaces/mamba-370m'
#     'state-spaces/mamba-130m'
pretrained_model_name = 'state-spaces/mamba-130m'

model = Mamba.from_pretrained(pretrained_model_name)
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')

## (2) Generate Text

In [3]:
import torch
import torch.nn.functional as F


def generate(model,
             tokenizer,
             prompt: str,
             n_tokens_to_gen: int = 50,
             sample: bool = True,
             top_k: int = 40):
    model.eval()
    
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids
    
    for token_n in range(n_tokens_to_gen):
        with torch.no_grad():
            indices_to_input = input_ids
            next_token_logits = model(indices_to_input)[:, -1]
        
        probs = F.softmax(next_token_logits, dim=-1)
        (batch, vocab_size) = probs.shape
        
        if top_k is not None:
            (values, indices) = torch.topk(probs, k=top_k)
            probs[probs < values[:, -1, None]] = 0
            probs = probs / probs.sum(axis=1, keepdims=True)
        
        if sample:
            next_indices = torch.multinomial(probs, num_samples=1)
        else:
            next_indices = torch.argmax(probs, dim=-1)[:, None]
        
        input_ids = torch.cat([input_ids, next_indices], dim=1)

    output_completions = [tokenizer.decode(output.tolist()) for output in input_ids][0]
    
    return output_completions

In [4]:
print(generate(model, tokenizer, 'Mamba is the'))

Mamba is the best on the market on CD and DVD. Some of my favorite albums released were "The Greatest Show" and "We're All Gonna Die". Some of my favorite releases are "Ruthless" by the late-80's band The


In [5]:
print(generate(model, tokenizer, 'John: Hi!\nSally:'))

John: Hi!
Sally: So you've heard this story.
John: I saw this, and it's about the
world's lowest building code,
and you go over there and you
go to the building codes
page, and you go
through the pages


In [6]:
print(generate(model, tokenizer, 'The meaning of life is '))

The meaning of life is ḫēfī ānā ānā.
A 'life' is a personal expression of a person's capacity to live. It
can be explained through the idea of 'life'; the life of the
individual which is an


In [7]:
print(generate(model, tokenizer, 'def reverse_string('))

def reverse_string(string_to_reverse=True):
    return "".join(reverse_string(res) for res in self.strings_to_reverse(string_to_reverse=False))

# define the name and format string for
